In [15]:
import joblib
import json
import pandas as pd
import numpy as np

from sklearn.metrics import precision_score, average_precision_score, confusion_matrix, roc_auc_score, recall_score

In [16]:
model = joblib.load('../models/fraud_pipeline.joblib')

In [17]:
X_test = pd.read_csv('../data/X_test.csv')

In [18]:
with open('../models/metadata.json') as f:
    metadata = json.load(f)

threshold = metadata['threshold']

In [19]:
y_proba = model.predict_proba(X_test)[:,1]
y_true = X_test['Class']
y_predict = (y_proba >= threshold).astype(int)

In [20]:
precision = precision_score(y_true, y_predict)
pr_auc = average_precision_score(y_true, y_proba)
roc_auc = roc_auc_score(y_true, y_proba)
cm = confusion_matrix(y_true, y_predict)

print(f'PR-AUC: {pr_auc:.4f} (val: {metadata["val_metrics"]["pr_auc"]:.4f})')
print(f'Recall: {recall_score(y_true, y_predict):.4f} (val: {metadata["val_metrics"]["recall_at_precision"]:.4f})')
print(f'Precision: {precision:.4f} (val: 0.7 — целевой порог)')
print(f'ROC-AUC: {roc_auc:.4f}')
print(cm)

PR-AUC: 0.8734 (val: 0.8431)
Recall: 0.8673 (val: 0.8283)
Precision: 0.6693 (val: 0.7 — целевой порог)
ROC-AUC: 0.9652
[[56822    42]
 [   13    85]]


In conclusion let's consider results shown by our model. First thing is, PR-AUC and recall are even higher on test(0.8734 and 0.8673) than on validation(0.8431 and 0.8283) that confirms that our model was not overfitted. Second, we should discuss Precision value that is 0.6696 vs claimed 0.7. We have 42 + 85 = 127 positive responses. Calculating standart error we get \sqrt(0.669 * 0.331 / 127) = 0.0417 => confidence interval is [0.59, 0.75] what contains 0.7. Comparing our result with [paper](https://www.iieta.org/download/file/fid/172943) working on the same dataset, we have the same recall, and superior PR-AUC (0.873 vs 0.867).